# 🧪 Labwright — the AI bench copilot that gets your numbers right

**Run a wet-lab experiment design and get every number *verified* — not hallucinated.**

Labwright flips the usual LLM role:
> **The model proposes raw inputs. The calculators produce and verify every derived number. The model cannot write a number the calculators didn't check.**

This notebook walks the full pipeline: install → design a perfused liver-chip → inspect the verified SOP → reverse-verify a published protocol's numbers.

* Project: https://github.com/qgeng1465/labwright
* Benchmark (bare LLM vs Labwright): 0.000 hallucination rate vs 33–58 % on `deepseek-v4-flash`/`-pro`.
* Run in **Colab** → **Runtime** → **Run all**. You'll be asked for a DeepSeek API key (any OpenAI-compatible key works).

In [ ]:
!pip install -q --upgrade "git+https://github.com/qgeng1465/labwright.git#egg=labwright[agent]"
print("installed")

## 1 · API key

Labwright defaults to DeepSeek (`deepseek-v4-flash`, thinking disabled — the arithmetic lives in the calculators, not the model). Any OpenAI-compatible API works via `LABWRIGHT_MODEL` / `LABWRIGHT_BASE_URL`. Get a key at https://platform.deepseek.com and paste it below (stored only in this session's environment).

In [ ]:
import getpass, os
key = getpass.getpass("DeepSeek API key (sk-...): ")
os.environ.setdefault("DEEPSEEK_API_KEY", key)
print("key set:", bool(os.environ["DEEPSEEK_API_KEY"]))

## 2 · Design a perfused liver-chip

Describe the experiment in plain words. Labwright's agent picks the goal, geometry and assumptions; the **calculators** compute shear, Reynolds, pressure drop, seeding, DMSO carry-over and replicate counts; the **verifier** re-derives every number before the design is accepted.

In [ ]:
from labwright.agent import LLMClient, DesignAgent
from labwright.sop import design_to_sop

def run_design(goal: str):
    llm = LLMClient()
    result = DesignAgent(llm).run(goal)
    print("status:", result.status)
    if result.verification_summary:
        print(result.verification_summary)
    if result.design:
        print("\n" + "=" * 60 + "\n" + design_to_sop(result.design))
    return result

goal = (
    "Design a perfused liver-chip experiment to model drug-induced liver injury, "
    "targeting sinusoidal wall shear (0.05 Pa), HepG2 seeded at 5e4 cells/cm^2, "
    "an APAP dose with DMSO vehicle control, and enough replicates to detect a 1-sigma "
    "effect with 80% power at alpha 0.05."
)
result = run_design(goal)

## 3 · See the verified design JSON

Every derived field here came from a deterministic calculator and passed the verifier — the model *could not* have typed these numbers from memory.

In [ ]:
import json
if result.design:
    print(json.dumps(result.design.model_dump(mode="json"), indent=2, ensure_ascii=False))

## 4 · Reverse-verify a *published* protocol (no API needed)

The reverse of design: take a paper's reported geometry, flow and its claimed shear, and check the claim follows from the paper's *own* inputs. This runs on pure calculators — no LLM, no API key.

Here a protocol claims **0.5 Pa** wall shear for a 400×100 µm × 20 mm channel at 2 µL/min. The physics says 0.05 Pa. The claim is a 10× unit mix-up (dyn/cm² vs Pa). Labwright recomputes and flags it.

In [ ]:
from labwright.published import verify_published_protocol

result = verify_published_protocol(
    chip={"width_um": 400, "height_um": 100, "length_mm": 20},
    flow={"flow_rate_uLmin": 2.0, "viscosity_pas": 1e-3, "density_kgm3": 1000},
    claimed={"shear_pa": 0.5},   # the paper's claim — 10x too high
    reference="Controlled demo case (not a real publication)",
)
print("status:", result["status"], "| discrepancies:", result["n_discrepancies"])
for check in result["checks"]:
    print(f"  {check['field']:<18} computed {check['computed']:<12} claimed {str(check['claimed']):<12} -> {check['verdict']}")

## What next?

* **CLI**: `labwright design "lung-on-chip at alveolar-capillary shear (~0.03 Pa)"` — the same pipeline, locally.
* **Batch literature check**: `python -m eval.run_verify_batch` re-verifies a set of published protocols.
* **Benchmark**: `python -m eval.report results/eval_flash.json` reproduces the numbers in the README.

Labwright is an experimental-design aid, not medical-device software. Always validate generated protocols against your lab's SOPs.